# 16 — Freddie companion model development

Develop separate Logistic Regression and XGBoost models for the reviewed **22-feature primary**
specification and **24-feature FICO sensitivity**. Use Freddie only: train on 2015–2016,
select on 2017, then refit on 2015–2017. Preserve the existing 29-feature models and study populations.

Run from a fresh kernel with **Run All** after pulling the notebook and `src/freddie_companion.py`.
The existing project environment needs the packages in `requirements.txt`.
This notebook does not open Fannie's raw CSV or candidate Parquet, read 2018–2023 outcome rows,
or calculate external performance. It reads only earlier aggregate contract metadata from Fannie.

The missing original download date is a provenance note, not a training gate. Cross-agency
predictor/outcome interpretation is reviewed before external scoring. Training the FICO companion
does not certify the Fannie score proxy. These are retrospective development experiments,
not models claimed to have been deployable in January 2018.

## 1. Check the original artifacts and reviewed specifications

Use the existing baseline manifest and contracts as read-only evidence. Automatic discovery accepts
only identical copies of the reviewed ZIP or metadata; set an override if different versions exist.
Generated outputs go into a new timestamped folder under `data/processed/Freddie_Mac/companion_models_2015_2017/`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, xgboost, joblib
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.freddie_features import (
    REQUIRED_SOURCE_COLUMNS, PREDICTOR_COLUMNS, build_engineered_features,
    fit_preprocessing_contract,
)
from src.freddie_companion import (
    TARGET, SEED, PRIMARY, SENSITIVITY, CATEGORICAL, SPECIFICATIONS, GRID, require,
    build_companion_features, fit_contract, apply_contract, model_pipeline, fit_model,
    calibration_table, probability_metrics, development_bin_edges, train_candidates,
)

DATA = ROOT / 'data/processed/Freddie_Mac'
BASE = DATA / 'model_baselines_2015_2017'
INPUT = DATA / 'freddie_modeling_dataset_2015_2023_24m.parquet'
FANNIE = ROOT / 'data/processed/Fannie_Mae'
REVIEW_14G = None  # Optional exact path to notebook_14g_final_compatibility_review.zip.
METADATA_15 = None  # Optional exact path to Notebook 15's metadata.json.
SOURCE_BASELINE = '39b199fe50fe27724b6b738f35c7683d8c04b841'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = DATA / 'companion_models_2015_2017' / RUN_ID

def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def notebook_source_sha(path):
    notebook = json.loads(Path(path).read_text(encoding='utf-8'))
    payload = [(cell['cell_type'], ''.join(cell['source'])) for cell in notebook['cells']]
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False).encode('utf-8')).hexdigest()

def locate(override, pattern):
    paths = [Path(override)] if override else sorted(FANNIE.glob(pattern))
    require(bool(paths) and all(p.is_file() for p in paths), f'Missing reviewed input: {pattern}')
    require(len({sha(p) for p in paths}) == 1, f'Different versions found; set the override for {pattern}.')
    return paths[-1]

require(INPUT.is_file(), f'Missing modeling dataset: {INPUT}')
baseline_path = BASE / 'model_manifest.json'
baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
original_contract_path = BASE / 'freddie_feature_contract.json'
original_contract = json.loads(original_contract_path.read_text(encoding='utf-8'))
require(sha(original_contract_path) == baseline['feature_contract_sha256'], 'Original contract hash differs.')
require(baseline['predictors'] == PREDICTOR_COLUMNS and len(PREDICTOR_COLUMNS) == 29, 'Original schema differs.')
require(baseline['target'] == TARGET and baseline['horizon_months'] == 24, 'Original target differs.')
require(baseline['development_years'] == [2015, 2016, 2017], 'Original development years differ.')
require(baseline['internal_fit_years'] == [2015, 2016] and baseline['internal_validation_year'] == 2017,
        'Original development split differs.')
require(original_contract['training_years'] == [2015, 2016, 2017], 'Original preprocessing years differ.')
require(original_contract['minimum_category_count'] == 100, 'Original rare-category policy differs.')
protected = [baseline_path, original_contract_path, ROOT / 'src/freddie_features.py']
for family, item in baseline['models'].items():
    require(Path(item['artifact']).name == item['artifact'], 'Invalid original artifact filename.')
    path = BASE / item['artifact']
    require(path.is_file() and sha(path) == item['sha256'], f'Original {family} artifact hash differs.')
    protected.append(path)
protected_before = {str(p.relative_to(ROOT)): sha(p) for p in protected}
source_before = {name: sha(ROOT / name) for name in ['src/freddie_features.py', 'src/freddie_companion.py']}
notebook_path = ROOT / 'notebooks/16_freddie_fannie_companion_model_development.ipynb'
notebook_source_before = notebook_source_sha(notebook_path)
input_sha256 = sha(INPUT)
require(input_sha256 == baseline['input_file_sha256'], 'Modeling dataset differs from baseline input.')

review_14g = locate(REVIEW_14G, '**/notebook_14g_final_compatibility_review.zip')
metadata_15_path = locate(METADATA_15, 'notebook_15_external_dataset_construction/*/metadata.json')
with zipfile.ZipFile(review_14g) as archive:
    proposal = json.loads(archive.read('candidate_contract.json'))
meta15 = json.loads(metadata_15_path.read_text(encoding='utf-8'))
require(proposal['frozen_contract_sha256'] == sha(original_contract_path), '14G original contract differs.')
require(meta15['frozen_contract_sha256'] == sha(original_contract_path), '15 original contract differs.')
require(meta15['review_14g_sha256'] == sha(review_14g), '15 references a different 14G review.')
require(PRIMARY == proposal['original_predictors_retained'] + proposal['additional_candidate'], '14G proposal differs.')
require(PRIMARY == meta15['primary_features'] and SENSITIVITY == meta15['fico_sensitivity_features'],
        '15 predictor order differs.')
require(len(PRIMARY) == 22 and len(SENSITIVITY) == 24 and len(set(SENSITIVITY)) == 24,
        'Companion predictor counts differ.')
require(len(CATEGORICAL) == 9 and len(PRIMARY) - len(CATEGORICAL) == 13, 'Companion types differ.')
print('Reviewed specifications verified. No Fannie feature rows or labels loaded.')
print('New output folder:', OUT)

## 2. Rebuild Freddie development features before imputation

Read only 2015–2017 rows from the validated modeling Parquet. Preserve the existing composite
target, eligibility, HARP population and cleaning. Rebuild origination features from the original
source columns; do not use Notebook 07's already imputed feature Parquet for validation.

The companion recodes Freddie P to PURCHASE and C/N to REFINANCE. Units and loan term have
canonical integer category spelling so values such as `1.0` and `1` represent the same category.
No additional loan exclusions, ratio caps or Fannie-sized subsample are introduced.

In [ ]:
columns = list(dict.fromkeys([*REQUIRED_SOURCE_COLUMNS, 'HORIZON_MONTHS']))
development = pd.read_parquet(INPUT, engine='pyarrow', columns=columns,
                              filters=[('SOURCE YEAR', 'in', [2015, 2016, 2017])])
require(set(development['SOURCE YEAR'].unique()) == {2015, 2016, 2017}, 'Development filter failed.')
require(not development[['SOURCE YEAR', 'LOAN IDENTIFIER']].isna().any().any(), 'Missing development key.')
require(not development.duplicated(['SOURCE YEAR', 'LOAN IDENTIFIER']).any(), 'Duplicate development key.')
require(pd.to_numeric(development['HORIZON_MONTHS']).eq(24).all(), 'Horizon differs.')
require(pd.to_numeric(development['ORIGINATION YEAR']).eq(development['SOURCE YEAR']).all(),
        'Origination and source years differ.')
require(development[TARGET].isin([0, 1]).all(), 'Invalid original composite target.')
require(len(development) == baseline['development_rows'] == 143598, 'Development population count differs.')
require(int(development[TARGET].sum()) == baseline['development_events'], 'Original development events differ.')
original_engineered = build_engineered_features(development)
require(fit_preprocessing_contract(original_engineered, training_years=(2015, 2016, 2017)) == original_contract,
        'Original preprocessing cannot be reproduced from this development input.')
engineered = build_companion_features(development)
pd.testing.assert_series_equal(engineered[TARGET], original_engineered[TARGET], check_names=True)
require(len(engineered) == len(development), 'Companion construction changed population membership.')
population = (engineered.groupby('SOURCE YEAR')[TARGET].agg(n='size', events='sum').reset_index())
population['prevalence'] = population.events / population.n
for years in ([2015, 2016], [2017]):
    require(set(engineered.loc[engineered['SOURCE YEAR'].isin(years), TARGET]) == {0, 1},
            'Both target classes are required in each split.')
availability = pd.DataFrame([dict(predictor=c, missing=int(engineered[c].isna().sum()),
                                  distinct_nonmissing=int(engineered[c].nunique()),
                                  specification='sensitivity_only' if c.startswith('FICO_') else 'primary_and_sensitivity')
                             for c in SENSITIVITY])
display(population)
display(availability)
del original_engineered, development

## 3. Train candidates and select using 2017 only

Use Notebook 10's three Logistic Regression and three XGBoost candidates for each specification.
Select within each specification and family: consider candidates less than 0.005 below the best
2017 Average Precision, then choose the lowest Brier score rounded to six decimals; ties use
the declared grid order. The rule and grid are fixed before this run's results.

Fit medians and the count-100 rare-category rules on 2015–2016 only. The encoder and Logistic
Regression scaler are also fitted on those years. Apply the stored rules to 2017.
No class weighting, resampling, early stopping, recalibration or threshold selection is added.
2017 is a model-selection set, so its reported metrics are internal selection results.

In [ ]:
(candidate_metrics, selected, internal_contracts,
 validation_predictions, validation_edges) = train_candidates(engineered)
require(len(candidate_metrics) == 12 and len(selected) == 4, 'Incomplete candidate development.')
selected_metrics = pd.DataFrame(selected.values()).drop(columns='brier_6').reset_index(drop=True)
display(candidate_metrics.drop(columns='params').round(6))
display(selected_metrics[['specification', 'family', 'params', 'roc_auc', 'average_precision', 'brier',
                          'prevalence', 'mean_predicted', 'risk_gap', 'ece_fixed_10']].round(6))

## 4. Inspect selected models' 2017 calibration

Use prediction deciles from the selected models' **2015–2016 training predictions** for the
2017 calibration plots. These edges are frozen without examining the 2017 prediction distribution.
Collapse tied edges. Save a separate ten-bin equal-width table for the reported ECE.
Empty bins retain undefined means. Calibration is descriptive and does not change model selection.

In [ ]:
y_val = engineered.loc[engineered['SOURCE YEAR'].eq(2017), TARGET].to_numpy(dtype=int)
calibration_parts = []
fixed_calibration_parts = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for axis, spec in zip(axes, SPECIFICATIONS):
    upper = .02
    for family in GRID:
        p = validation_predictions[(spec, family)]
        bins = calibration_table(y_val, p, validation_edges[(spec, family)])
        calibration_parts.append(bins.assign(specification=spec, family=family,
                                             binning='2015_2016_training_prediction_deciles'))
        fixed_calibration_parts.append(calibration_table(y_val, p).assign(specification=spec, family=family))
        occupied = bins.loc[bins.n.gt(0)]
        axis.plot(occupied.mean_predicted, occupied.observed_rate, marker='o', label=family)
        upper = max(upper, float(occupied[['mean_predicted', 'observed_rate']].max().max()))
    axis.plot([0, upper], [0, upper], '--', color='gray', label='perfect calibration')
    axis.set(title=spec, xlabel='Mean predicted risk', ylabel='Observed event rate')
    axis.legend()
fig.suptitle('Freddie 2017 internal validation — training-defined prediction bins')
fig.tight_layout()
calibration_validation = pd.concat(calibration_parts, ignore_index=True)
fixed_calibration_validation = pd.concat(fixed_calibration_parts, ignore_index=True)
for _, part in calibration_validation.groupby(['specification', 'family']):
    require(int(part.n.sum()) == len(y_val) and int(part.events.sum()) == int(y_val.sum()),
            'Calibration totals differ.')
display(calibration_validation.loc[calibration_validation.n.gt(0)].round(6))
plt.show()

## 5. Refit on 2015–2017 and freeze separate artifacts

Refit preprocessing and each selected model on the complete Freddie development population.
Save internal and final contracts separately. Check serialized model predictions against the
in-memory models, and save development-defined calibration edges for later evaluation.
Development reference metrics below are in-sample context, not external performance.

Every run uses a new folder. The final manifest is written only after artifact, input and
original-baseline preservation checks pass. A folder without that manifest is incomplete.

In [ ]:
OUT.mkdir(parents=True, exist_ok=False)
(OUT / 'figures').mkdir()
fig.savefig(OUT / 'figures/validation_2017_calibration.png', dpi=160, bbox_inches='tight')
y_all = engineered[TARGET].to_numpy(dtype=int)
models = {}
reference_metrics = []
reference_bins = {}
reference_calibration = []
for spec in SPECIFICATIONS:
    final_contract = fit_contract(engineered, spec, [2015, 2016, 2017])
    contract_file = OUT / f'{spec}_feature_contract.json'
    contract_file.write_text(json.dumps(final_contract, indent=2, allow_nan=False), encoding='utf-8')
    (OUT / f'{spec}_internal_contract.json').write_text(
        json.dumps(internal_contracts[spec], indent=2, allow_nan=False), encoding='utf-8')
    read_contract = json.loads(contract_file.read_text(encoding='utf-8'))
    require(read_contract == final_contract, 'Contract read-back differs.')
    x_all = apply_contract(engineered, read_contract)
    require(list(x_all) == SPECIFICATIONS[spec], 'Final feature order differs.')
    for family in GRID:
        key = (spec, family)
        chosen = selected[key]
        params = json.loads(chosen['params'])
        pipe = fit_model(model_pipeline(family, params, read_contract), x_all, y_all)
        artifact = OUT / f'{spec}_{family}_2015_2017.joblib'
        joblib.dump(pipe, artifact)
        loaded = joblib.load(artifact)
        probe = pd.concat([x_all.head(50), x_all.tail(50)]).drop_duplicates()
        np.testing.assert_allclose(pipe.predict_proba(probe), loaded.predict_proba(probe), rtol=1e-12, atol=1e-12)
        require(list(loaded.feature_names_in_) == SPECIFICATIONS[spec], 'Serialized feature order differs.')
        p = loaded.predict_proba(x_all)[:, 1]
        model_key = f'{spec}__{family}'
        edges = development_bin_edges(p)
        reference_bins[model_key] = edges
        reference_calibration.append(calibration_table(y_all, p, edges).assign(
            specification=spec, family=family, interpretation='development_in_sample'))
        reference_metrics.append(dict(specification=spec, family=family,
                                      interpretation='development_in_sample', **probability_metrics(y_all, p)))
        models[model_key] = dict(specification=spec, family=family, selected_params=params,
                                selected_candidate_order=int(chosen['candidate_order']),
                                artifact=artifact.name, sha256=sha(artifact),
                                feature_contract=contract_file.name, feature_contract_sha256=sha(contract_file),
                                predictors=list(SPECIFICATIONS[spec]),
                                transformed_feature_count=len(loaded.named_steps['prepare'].get_feature_names_out()))
        print('Frozen:', artifact.name, flush=True)
require(len(models) == 4, 'Final model set is incomplete.')
pd.DataFrame(reference_metrics).to_csv(OUT / 'development_reference_metrics.csv', index=False)
pd.concat(reference_calibration, ignore_index=True).to_csv(OUT / 'development_reference_calibration.csv', index=False)
(OUT / 'frozen_calibration_bins.json').write_text(json.dumps(reference_bins, indent=2), encoding='utf-8')

## 6. Save the review bundle and verify preservation

The review ZIP contains aggregate tables, plots, feature contracts, model settings, and file hashes.
Model artifacts and all source data stay local. It contains no loan identifiers, individual feature
rows, labels or predictions. Compare the two specifications descriptively; do not choose which
external result to report after inspecting Fannie outcomes.

In [ ]:
candidate_metrics.to_csv(OUT / 'development_2017_candidate_metrics.csv', index=False)
selected_metrics.to_csv(OUT / 'development_2017_selected_metrics.csv', index=False)
calibration_validation.to_csv(OUT / 'validation_2017_calibration_bins.csv', index=False)
fixed_calibration_validation.to_csv(OUT / 'validation_2017_fixed_probability_bins.csv', index=False)
(OUT / 'internal_validation_calibration_bins.json').write_text(
    json.dumps({f'{s}__{f}': edges for (s, f), edges in validation_edges.items()}, indent=2), encoding='utf-8')
population.to_csv(OUT / 'training_population_summary.csv', index=False)
availability.to_csv(OUT / 'predictor_availability.csv', index=False)
sources = {name: sha(ROOT / name) for name in ['src/freddie_features.py', 'src/freddie_companion.py']}
require(sources == source_before, 'A feature module changed during the run; restart and rerun.')
require(notebook_source_sha(notebook_path) == notebook_source_before,
        'Notebook source changed during the run; restart and rerun.')
require({str(p.relative_to(ROOT)): sha(p) for p in protected} == protected_before,
        'An original baseline file or feature source changed during this run.')
require(sha(INPUT) == input_sha256, 'Modeling input changed during this run.')
feature_definitions = dict(
    primary=list(PRIMARY), fico_sensitivity=list(SENSITIVITY),
    purchase_refinance={'P': 'PURCHASE', 'C': 'REFINANCE', 'N': 'REFINANCE'},
    continuous_transformations='Reuse existing cleaned Freddie source and original engineering.',
    integer_category_spelling=['NUMBER_OF_UNITS', 'ORIGINAL_LOAN_TERM'],
    rare_category_minimum_count=100, missing_category='MISSING', unknown_category='OTHER',
    unseen_onehot_category='ignore', fico_external_equivalence='conditional_sensitivity_only',
    no_new_population_filters=True, no_harp_exclusion=True,
)
(OUT / 'companion_feature_definitions.json').write_text(json.dumps(feature_definitions, indent=2), encoding='utf-8')
manifest = dict(
    stage='16', status='development_complete_pending_review', created_utc=datetime.now(timezone.utc).isoformat(),
    run_id=RUN_ID, evidence_baseline_commit=SOURCE_BASELINE, target=TARGET, horizon_months=24,
    target_interpretation='Existing Freddie composite, including RA; cross-agency mapped outcome reviewed separately.',
    development_years=[2015, 2016, 2017], internal_fit_years=[2015, 2016], internal_validation_year=2017,
    development_rows=len(engineered), development_events=int(y_all.sum()),
    specifications={k: list(v) for k, v in SPECIFICATIONS.items()}, models=models,
    candidate_grid=GRID, selection_rule='Within <0.005 of best AP; lowest six-decimal Brier; declared grid order.',
    validation_interpretation='Internal model-selection results, not independent test estimates.',
    ece_definition='Ten fixed equal-width probability bins, weighted absolute calibration error.',
    calibration_bin_policy='Quantiles of final Freddie 2015-2017 predictions; ties collapsed; endpoints 0 and 1.',
    random_seed=SEED, input_file_sha256=input_sha256, source_sha256=sources,
    notebook_source_sha256=notebook_source_before, original_artifacts_sha256=protected_before,
    review_14g_sha256=sha(review_14g), metadata_15_sha256=sha(metadata_15_path),
    fannie_candidate_sha256=meta15['candidate_dataset_sha256'],
    fannie_source_sha256=meta15['source_sha256'],
    fannie_download_date='not_retained', fannie_publication_release='unverified_provenance_note',
    fannie_feature_rows_loaded=False, fannie_outcomes_loaded=False, out_of_time_rows_loaded=False,
    external_scored=False, external_evaluation_approved=False,
    original_artifacts_preserved=True,
    python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
    scikit_learn=sklearn.__version__, xgboost=xgboost.__version__, joblib=joblib.__version__,
)
notes = """# Notebook 16 review

Four separate Freddie companion models were fitted and frozen. Twelve internal candidates
were compared using the predeclared 2017 rule. Inspect selected metrics and calibration before
external evaluation. Development reference metrics are in-sample.

The FICO variant remains a sensitivity. Exact Fannie download date/publication release was not
retained; this is a provenance limitation, not a failed structural check or automatic scoring block.
Confirm predictor mapping, composite-outcome interpretation, primary/strict Fannie eligibility,
and the unequal sampled/acquisition-file populations before external evaluation.

No new population filters, target definitions, reweighting, recalibration, original artifact changes,
Fannie model selection or external scoring were performed. Notebook 17 has not been started.
"""
(OUT / 'review_notes.md').write_text(notes, encoding='utf-8')
# The manifest's presence marks a complete run; include every preceding output fingerprint.
manifest['output_sha256'] = {str(p.relative_to(OUT)): sha(p) for p in sorted(OUT.rglob('*')) if p.is_file()}
(OUT / 'model_manifest.json').write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
require(json.loads((OUT / 'model_manifest.json').read_text(encoding='utf-8')) == manifest, 'Manifest read-back differs.')
review_zip = OUT / 'notebook_16_companion_development_review.zip'
with zipfile.ZipFile(review_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix in {'.csv', '.json', '.png', '.md'}:
            archive.write(p, str(p.relative_to(OUT)))
with zipfile.ZipFile(review_zip) as archive:
    require(not any(name.endswith(('.joblib', '.parquet', '.sqlite')) for name in archive.namelist()),
            'Review ZIP includes a local-only artifact.')
print('Original artifacts preserved; four companion artifacts verified.')
print('Output folder:', OUT)
print('UPLOAD THIS FILE:', review_zip)

## Next review

Save this executed notebook. Upload **`notebook_16_companion_development_review.zip`** from
the folder printed above. Keep the four `.joblib` files and contracts in their timestamped folder;
later evaluation will load that exact completed run. No original notebook needs to be rerun.

Review internal metrics and calibration, then finalize the external evaluation interpretation
before Notebook 17. The original Fannie download date is not required to execute this notebook.